# Recomendador v1.2: de una ficha en PDF a productos Interlub

**Antes de correr:** extrae el PDF con el prompt de `scripts/prompts/extraccion_pdfs.md` y guarda el JSON
(solo la lista `[...]`) en `data/processed/`. Luego ajusta la celda de configuración, corre todo en orden y
**guarda el notebook** (así las salidas quedan en el archivo).

Cómo decide (detalle y tolerancias en `scripts/recomendador.py`):
1. **Filtros**: misma familia · el rango de temperatura de Interlub cubre el de la ficha (le pueden faltar hasta
   5 °C por arriba y 10 °C por abajo) · si la ficha es grado alimenticio, solo productos NSF.
2. **Parecido técnico** (0 a 1), con tolerancias técnicas:
   - **temperatura**: cubrir todo el rango ✓. Quedarse corto por **arriba** castiga más (cada 10 °C faltantes
     bajan la similitud a la mitad, porque la vida del producto se reduce a la mitad); por abajo baja
     linealmente (20 °C faltantes = 0).
   - **viscosidad**: ±10 % = mismo grado ISO ✓; un grado ISO de diferencia ~; dos o más ✗.
   - **NLGI**: medio grado de diferencia ~; uno o más ✗.
   - **desempeño** (punto de gota, soldadura y desgaste 4 bolas, Timken…): si Interlub es igual o mejor ✓;
     si es peor, castiga (p. ej. soldadura un escalón abajo, 620 → 500, queda ~).
   - **espesante** y **aceite base**: suman si coinciden.
3. **Advertencias** (no cambian el puntaje): si cambia el espesante o el aceite base, avisa que hay que limpiar
   o probar la compatibilidad antes de mezclar.
3. **Cobertura**: qué fracción de los datos de la ficha se pudo comparar.
4. **Propiedades críticas** (NLGI, viscosidad, temperatura): además multiplican el puntaje (✓ ×1, ~ ≈ ×0.75,
   ✗ ×0.5), para que un ✗ en ellas no se compense con ✓ en lo demás.
   `puntaje = parecido × √cobertura × factor_critico`
5. **Texto** (opcional): similitud de embeddings, solo informativa (no cambia el orden).

In [ ]:
# Configuración
JSON = "../data/processed/prueba.json"   # el JSON que devolvió la IA
PRODUCTO = "ULTRAPLEX MCS 2"             # nombre_producto a usar como consulta
K = 5                                     # cuántas recomendaciones
USAR_TEXTO = False                        # mostrar similitud de embeddings (informativa, no ordena)

In [ ]:
# fichas_pdf.py y recomendador.py viven en scripts/
import sys

sys.path.insert(0, "../scripts")

In [ ]:
# Recarga automática: si cambia recomendador.py o fichas_pdf.py, el notebook usa la versión nueva
# sin reiniciar el kernel.
%load_ext autoreload
%autoreload 2

import json

import fichas_pdf as F
import numpy as np
import pandas as pd
import recomendador as R

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

interlub = pd.read_csv("../data/processed/DATOS_TEC_clean.csv")
with open(JSON, encoding="utf-8") as f:
    fichas = F.normalizar(json.load(f))
fichas[["nombre_producto", "marca", "familia", "espesante", "grado_nlgi_por_penetracion",
        "temperatura_de_servicio_c_min", "temperatura_de_servicio_c_max"]]

In [ ]:
# La ficha que se usa como consulta: solo las columnas con dato
ficha = fichas.loc[fichas["nombre_producto"] == PRODUCTO].iloc[0]
ficha.drop(["descripcion", "aplicaciones", "beneficios_caracteristicas", "certificaciones", "otras_propiedades"]).dropna()

In [ ]:
# Opcional: similitud de texto (embeddings de la ficha en texto vs las de Interlub)
sim_texto = None
if USAR_TEXTO:
    from sentence_transformers import SentenceTransformer

    emb = np.load("../data/processed/embeddings.npz", allow_pickle=True)
    assert (emb["interlub_id"] == interlub["codigoproducto"].to_numpy()).all(), "re-corre 02_pdfs.ipynb"
    modelo = SentenceTransformer(str(emb["modelo"]))
    sim_texto = R.similitud_texto(ficha, interlub, modelo, emb["interlub_ficha"])

In [ ]:
recs, avisos = R.recomendar(ficha, interlub, k=K, sim_texto=sim_texto)
for a in avisos:
    print("•", a)
recs

In [ ]:
# Comparación lado a lado: la ficha vs cada recomendado
R.comparar(ficha, interlub, recs)

## Cómo leer el resultado
- **puntaje**: el número para ordenar. **parecido_tecnico**: qué tan parecidos son en lo que sí se pudo comparar.
  **cobertura**: qué fracción de los datos de la ficha tenía también el producto Interlub (1 = todo).
  **factor_critico**: 1 si NLGI, viscosidad y temperatura están bien; baja con cada ~ o ✗ en ellas.
- **por_que**: cada propiedad como `ficha vs Interlub` con ✓ (igual o mejor), ~ (algo distinto o un poco peor)
  o ✗ (muy distinto o claramente peor). Un `>` antes del valor de la ficha significa "al menos"
  (p. ej. punto de gota > 300). En temperatura dice cuántos °C le faltan al producto Interlub, si le faltan.
- Un puntaje alto con cobertura baja es poco confiable: se comparó en pocas propiedades.
- Revisa los **avisos**: dicen qué filtros se aplicaron y qué datos de la ficha no se pudieron usar.